# 01 — Exploratory Data Analysis

Retail store inventory (`data/retail_store_inventory.csv`): daily store-product rows with sales, on-hand stock, pricing, weather, and a vendor demand forecast.

This notebook profiles the table and records the data-quality facts that drive the modeling design.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_dataset, dataset_profile

sns.set_theme(style="whitegrid")
df = load_dataset()
profile = dataset_profile(df)
profile


In [ ]:
df.head()


In [ ]:
df.describe().T


In [ ]:
df.isnull().sum()


## Shape of the panel

Five stores × 20 products × 731 dates. The file covers 2022-01-01 through 2024-01-01, not a single calendar year.


In [ ]:
print(df.groupby(["Store ID", "Product ID"]).size().describe())
print("stores", df["Store ID"].nunique(), "products", df["Product ID"].nunique())
for col in ["Category", "Region", "Weather Condition", "Seasonality"]:
    print(col, df[col].value_counts().to_dict())


## Daily volume and the inventory cap


In [ ]:
daily = df.groupby("Date")["Units Sold"].sum()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily.values)
ax.set_title("Total units sold by day")
ax.set_ylabel("Units")
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
sample = df.sample(6000, random_state=42)
ax.scatter(sample["Inventory Level"], sample["Units Sold"], s=8, alpha=0.25)
lim = max(sample["Inventory Level"].max(), sample["Units Sold"].max())
ax.plot([0, lim], [0, lim], "r--", label="sold = inventory")
ax.set_xlabel("Inventory Level")
ax.set_ylabel("Units Sold")
ax.legend()
ax.set_title("Sales never exceed on-hand stock")
plt.show()


## Correlations

`Demand Forecast` is almost collinear with `Units Sold`. `Inventory Level` is the only other strong numeric signal. Price, discount, competitor price, and units ordered are essentially uncorrelated with sales. Competitor price itself is just price ± about $5.


In [ ]:
num = ["Units Sold", "Inventory Level", "Units Ordered", "Demand Forecast", "Price", "Discount", "Competitor Pricing", "Holiday/Promotion"]
print(df[num].corr().round(3))
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Numeric correlations")
plt.show()


## External factors barely move the mean


In [ ]:
display(df.groupby("Category")["Units Sold"].mean())
display(df.groupby("Region")["Units Sold"].mean())
display(df.groupby("Weather Condition")["Units Sold"].mean())
display(df.groupby("Holiday/Promotion")["Units Sold"].mean())
display(df.groupby("Seasonality")["Units Sold"].mean())


## Data-quality findings (modeling implications)

1. **Do not treat `Demand Forecast` as an ordinary feature in an "from scratch" model.** It is already a forecast of the target (corr ≈ 0.997, MAE ≈ 8). Using it is *forecast refinement*, which is valid, but it is not independent demand prediction.
2. **Lags will not help much.** Store-product lag-1 autocorrelation of units sold is ≈ 0.
3. **Category / region / seasonality labels are not entity attributes.** Each product appears in every category; each store appears in every region; the seasonality flag does not match calendar season.
4. **Units sold look like a random fraction of inventory** (mean sold / mean inventory ≈ 0.5, sold ≤ inventory always). Operational models that include inventory will capture that cap and little else.
5. **MAPE is a poor headline metric here.** 360 zero-sales rows make classic MAPE explode. Report WAPE, MAE, RMSE, R², and MAPE on demand ≥ 10 alongside it.
